# CommittedAgent vs Joint-RL 2P3G Comparison

Side-by-side comparison for `n=30` sessions, `12` 2P3G trials per session. Metrics use the same notebook-style definitions as the committed-agent analysis.


Definitions:
- Success: session-level mean success rate.
- Coordination efficiency: per-agent post-new-goal trajectory efficiency, successful new-goal trials only.
- Commitment: `firstDetectedGoal == finalReachedGoal` for each agent on new-goal trials.
- Signaling: first move after new-goal presentation moves closer to the reached goal but not the alternative.


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT_DIR = Path('dataAnalysis/analyses/outputs/comparisons/agent_pair_comparison_2p3g')
SOURCES = {
    'Committed Agents': {
        'summary': OUT_DIR / 'committed_vs_committed_2p3g_summary_sessions_0_to_29.json',
        'metrics': OUT_DIR / 'committed_vs_committed_2p3g_notebook_metrics_sessions_0_to_29.json',
    },
    'Joint-RL Agents': {
        'summary': Path('dataAnalysis/model_model/joint_rl/outputs/joint_rl_vs_joint_rl_simulation/joint_rl_vs_joint_rl_2p3g_summary_sessions_0_to_29.json'),
        'metrics': Path('dataAnalysis/model_model/joint_rl/outputs/joint_rl_vs_joint_rl_simulation/joint_rl_vs_joint_rl_2p3g_notebook_metrics_sessions_0_to_29.json'),
    },
}

def mean_ci(vals):
    vals = np.array([v for v in vals if pd.notna(v)], dtype=float)
    if len(vals) == 0:
        return np.nan, np.nan, np.nan, 0
    mean = float(vals.mean())
    sd = float(vals.std(ddof=1)) if len(vals) > 1 else 0.0
    se = sd / np.sqrt(len(vals)) if len(vals) else np.nan
    ci = 1.96 * se if np.isfinite(se) else np.nan
    return mean, mean - ci, mean + ci, int(len(vals))

def build_summary():
    rows = []
    for label, paths in SOURCES.items():
        summary = json.loads(paths['summary'].read_text())
        metrics = json.loads(paths['metrics'].read_text())
        row_df = pd.DataFrame(metrics['row_level_measures'])
        success_mean, success_lo, success_hi, success_n = mean_ci(
            [s['successRate'] for s in summary['sessionSummaries'] if s.get('successRate') is not None]
        )
        new_goal = row_df[row_df['newGoalPresented'] == True].copy()
        commit_participant = (
            new_goal.dropna(subset=['commitment'])
            .groupby(['participantId', 'partnerType'])['commitment']
            .mean()
            .reset_index()
        )
        commitment_mean, commitment_lo, commitment_hi, commitment_n = mean_ci(commit_participant['commitment'])
        eff = metrics['summary']['overall']['efficiency_success_only']
        sig = metrics['summary']['overall']['signaling_all_new_goal']
        rows.append({
            'agent_pair': label,
            'total_trials': summary['totalTrials'],
            'successful_trials': summary['successfulTrials'],
            'success_rate_raw': summary['successRate'],
            'success_mean': success_mean,
            'success_ci_lower': success_lo,
            'success_ci_upper': success_hi,
            'success_n': success_n,
            'commitment_rate_raw': summary['commitmentRate'],
            'commitment_mean': commitment_mean,
            'commitment_ci_lower': commitment_lo,
            'commitment_ci_upper': commitment_hi,
            'commitment_n': commitment_n,
            'efficiency_mean': eff['mean'] / 100,
            'efficiency_ci_lower': eff['ci_lower'] / 100,
            'efficiency_ci_upper': eff['ci_upper'] / 100,
            'efficiency_n': eff['n_participants'],
            'signaling_mean': sig['mean'],
            'signaling_ci_lower': sig['ci_lower'],
            'signaling_ci_upper': sig['ci_upper'],
            'signaling_n': sig['n_participants'],
            'new_goal_trials': summary['newGoalPresentedTrials'],
            'new_goal_agent_rows': metrics['summary']['new_goal_rows'],
        })
    return pd.DataFrame(rows)

summary_df = build_summary()
summary_df.to_csv(OUT_DIR / 'committed_vs_joint_rl_2p3g_summary.csv', index=False)
summary_df

In [ ]:
metric_specs = [
    ('Success Rate (%)', 'success'),
    ('Coordination Efficiency (%)', 'efficiency'),
    ('Commitment (%)', 'commitment'),
    ('Signaling Move (%)', 'signaling'),
]
colors = ['#5f88b6', '#d98c48']
labels = summary_df['agent_pair'].tolist()

plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(2, 2, figsize=(11, 8.5), dpi=150)
fig.suptitle('CommittedAgent vs Joint-RL: 2P3G Agent-Pair Comparison', fontsize=17, fontweight='bold', y=0.98)
for ax, (title, key) in zip(axes.flat, metric_specs):
    means = summary_df[f'{key}_mean'].to_numpy(dtype=float) * 100
    lows = summary_df[f'{key}_ci_lower'].to_numpy(dtype=float) * 100
    highs = summary_df[f'{key}_ci_upper'].to_numpy(dtype=float) * 100
    yerr = np.vstack([np.maximum(0, means - lows), np.maximum(0, highs - means)])
    x = np.arange(len(labels))
    bars = ax.bar(x, means, yerr=yerr, capsize=5, color=colors, alpha=0.92, width=0.58, edgecolor='none')
    ax.set_title(title, fontsize=13.5, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_ylim(0, 105)
    ax.set_yticks(np.arange(0, 101, 20))
    ax.set_ylabel('(%)', fontsize=10)
    ax.grid(axis='y', color='#c9c9c9', linewidth=1)
    ax.grid(axis='x', visible=False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, max(2, mean * 0.04), f'{mean:.1f}', ha='center', va='bottom', color='white', fontsize=8, fontweight='bold')
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(OUT_DIR / 'committed_vs_joint_rl_2p3g_4panel.png', bbox_inches='tight')
plt.show()